In [ ]:
%load_ext autoreload
%autoreload 2

# OakLang Agent HITL + Adaptative/Agentic RAG

## Agents/Graphs as simple tools

In [ ]:
## COMPILE THE GRAPHS
from core_examples.config.layouts.local_vectorstore_adaptive_rag_config_graph import (
    LocalVectorStoreAdaptiveRAGConfigGraph,
)
from core_examples.config.layouts.oak_human_loop_config_graph import (
    OakHumanLoopConfigGraph,
)
from core_examples.models.stategraph.ragstategraph import RAGState
from core_examples.models.stategraph.stategraph import SharedState
from core_examples.utils.logger import configure_logging
from frankstate.workflow_builder import WorkflowBuilder
from services.foundry.llms import LLMServices

## Setup logging Configuration
configure_logging()

LLMServices.launch()

## Workflow Configuration for the main graph
workflow_builder = WorkflowBuilder(
    config=LocalVectorStoreAdaptiveRAGConfigGraph,
    state_schema=RAGState,
)
LOCAL_VECTORSTORE_ADAPTIVE_RAG_GRAPH = workflow_builder.compile() # compile the graph

## Workflow Configuration for the main graph
workflow_builder = WorkflowBuilder(
    config=OakHumanLoopConfigGraph,
    state_schema=SharedState,
)
OAKLANG_AGENT_GRAPH = workflow_builder.compile() # compile the graph

In [ ]:
## BUILD THE SIMPLE TOOLS
from langchain.agents import create_agent
from langchain_core.tools import tool

from services.foundry.llms import LLMServices

LLMServices.launch()

@tool
async def handoff_oaklang_agent(input: str) -> str:
    """Tool to ask the evolution or random movements of certain Pokémon. The input is a question."""
    message_input = {"messages": [{"role": "human", "content": input}]}
    response = await OAKLANG_AGENT_GRAPH.ainvoke(message_input)
    return response['messages'][-1].content

@tool
async def adaptive_rag_tool(input: str) -> str:
    """Tool to use RAG about Pokémon series questions. The input is a question."""
    message_input = {"messages": [{"role": "human", "content": input}]}
    response = await LOCAL_VECTORSTORE_ADAPTIVE_RAG_GRAPH.ainvoke(message_input)
    return response['messages'][-1].content

tools = [handoff_oaklang_agent, adaptive_rag_tool]

agent_executor = create_agent(model = LLMServices.model, tools=tools, system_prompt="You are an agent that can route into tools")

async def main():
    return await agent_executor.ainvoke({
        "messages": [{
            "role": "human",
            "content": "In the third chapter of the Pokémon series, what is the name of the Pokémon that was flying in the daytime sky and subsequently caught? and what is the evolution of that pokemon?"
        }]
    })

response = await main()


In [ ]:
response

In [ ]:
response["messages"][-1].content

## Agents/Graphs via Local MCP tools

In [6]:
from langchain.agents import create_agent
from langchain_mcp_adapters.client import MultiServerMCPClient

from services.foundry.llms import LLMServices

LLMServices.launch()

client = MultiServerMCPClient(
    {
        "CustomMPCServer": {  # NOTE: python src/services/mcp/server_oaklang_agent.py
            "url": "http://localhost:8000/mcp",
            "transport": "http",
            "headers": {
                "Authorization": "Bearer YOUR_TOKEN",
                "X-Custom-Header": "admin"
            },
        },
        # "CustomMPCServer2": {
        #     "command": "python",
        #     "args": ["src/services/mcp/server_adaptive_rag.py"],
        #     "transport": "stdio",
        # },
    }
)
tools = await client.get_tools()
agent = create_agent(
    LLMServices.model,
    tools
)


response = await agent.ainvoke(
    {"messages": [{
        "role": "human",
        "content": "In the third chapter of the Pokémon series, what is the name of the Pokémon that was flying in the daytime sky and subsequently caught? and what is the evolution of that pokemon and random movements?"
        }]}
)


In [ ]:
tools

In [ ]:
response

In [ ]:
response["messages"][-1].content

## MPC Remote Server Function App

In [ ]:
from langchain.agents import create_agent
from langchain_mcp_adapters.client import MultiServerMCPClient

from services.foundry.llms import LLMServices

LLMServices.launch()

# # Locally # NOTE: need extra auth from local.settings
# client = MultiServerMCPClient(
#     {
#         "azure_function_mcp": {
#             "url": "http://localhost:8080/runtime/webhooks/mcp/sse?code=<function-app-keys-mcp-extension>",
#             "transport": "sse",
#         }
#     }
# )

# Remote
client = MultiServerMCPClient(
    {
        "remote_azure_function_mcp": {
            "url": "https://<function-app-name>.azurewebsites.net/runtime/webhooks/mcp/sse?code=<function-app-keys-mcp-extension>",
            "transport": "sse",
        }
    }
)


tools = await client.get_tools()
agent = create_agent(
    LLMServices.model,
    tools
)

response = await agent.ainvoke(
    {"messages": [{
        "role": "human",
        "content": 'Hi, could you use the calculator tool to check 1+3?'
    }]}
)
